In [ ]:
# %% [markdown]
# # Linear Track DLC → Cleaned, Binned Trajectories
#
# This notebook:
# - Loads a DeepLabCut CSV and a `dlc_trim_info.csv` from a session directory
# - Chooses a bodypart (either from `dlc_trim_info` or by robustness)
# - Trims the trajectory to the desired frame window
# - Cleans the 2D trajectory (remove low-likelihood points, remove big jumps, interpolate, smooth)
# - Labels which reward port (start / end) the animal is in
# - Segments trials as "leaving one port, arriving at the other"
# - Projects the 2D trajectory onto the main track axis (linearization)
# - Bins positions along the track
# - Saves CSVs with frame-wise data and per-trial linearized + binned positions

# %% 
import glob
import os
import re
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import h5py
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter1d
from scipy.signal import medfilt


In [ ]:
# =========================
# Global parameters
# =========================

FPS = 25.0                     # frames per second of DLC video
LIKELIHOOD_THRESHOLD = 0.9     # DLC likelihood threshold for "good" samples
MAX_JUMP = 5.0                # max pixel jump between consecutive frames before marking as bad
SMOOTHING_SIGMA = 1.0          # Gaussian smoothing sigma (in samples) for trajectory
SPEED_THRESHOLD = 2.0          # pixels / frame threshold for movement
MIN_SPEED_DURATION = 0.4       # minimum duration (s) for continuous movement
MIN_DWELL_FRAMES = 4          # frames required inside a port to "commit" to it
MIN_TRIAL_DURATION_S = 0.5     # minimum duration (s) of a trial to keep
MIN_DIR_COS = -0.5              # minimum mean cosine alignment with template
N_BINS = 70                    # number of spatial bins along the track

# Port ROIs, by camera era. The linear rig is physically fixed, so the aging cohort (2024) and
# the young cohort's pre-May-2026 sessions share one camera position; only Mouse944 and
# Mouse945-Round2 (>= 2026-05-01) moved to the other. See CLAUDE.md "Reward outcome" section and
# docs/reward_extraction_plan.md.
#
# Centers = the water-port LED pixel coordinates (verified against real video frames). Radius =
# 52.5px, sized from the animal's actual tracked nose position at the moment of reward (measured via
# detect_reward_flushes.py + DLC across several sessions per era): the nose sits 12-25px from the
# LED center plus ~10px frame-to-frame jitter, so 52.5px (35px + 50%% margin) comfortably contains it while staying far
# tighter than the old single hardcoded circle (radius 60, offset ~30-40px from the true port
# location) that was silently dropping trials whenever an approach fell outside it.
PORT_ROIS_BY_ERA = {
    # (xmin, xmax, ymin, ymax) -- manually calibrated against real video frames
    # using the "Decide on port ROI locations" cell below.
    "aging2024":   {"left": (185, 305, 615, 715), "right": (1690, 1810, 560, 660)},
    "pre2026-05":  {"left": (160, 280, 605, 705), "right": (1700, 1820, 550, 650)},
    "post2026-05": {"left": (165, 285, 665, 765), "right": (1700, 1820, 600, 700)},
}


def era_for_session(session_dir) -> str:
    """Camera era for a LINEAR-track session, from the timestamp in its directory name.

    Matches detect_reward_flushes.py's epoch_for_session() exactly, so the same session is always
    assigned the same era whether it goes through reward detection or trial segmentation.
    """
    m = re.search(r"(\d{4}-\d{2}-\d{2})T\d{2}_\d{2}_\d{2}", str(session_dir))
    if not m:
        raise ValueError(f"no timestamp found in session_dir: {session_dir}")
    date = m.group(1)
    if date < "2025-01-01":
        return "aging2024"
    return "post2026-05" if date >= "2026-05-01" else "pre2026-05"


# Backward-compatible default, used only by the single-session example cell below -- the batch
# loop further down looks up the correct era per session instead of relying on this constant.
PORT_ROIS_DEFAULT = PORT_ROIS_BY_ERA["pre2026-05"]


In [ ]:


# =========================
# Helper functions
# =========================

def extract_datetime_stamp(path: str) -> str:
    """
    Extracts a timestamp in several possible formats, for example:
        2024-06-06T14_35_51
        2024-06-06_14-35-51
        2024-06-06 14-35-51
        2024-06-06-14-35-51
        behavior2024-05-24T18_42_16DLC...
    Returns "unknown_datetime" if no match.
    """

    # 1. Strict full pattern first (YYYY-MM-DD + T + HH_MM_SS)
    patterns = [
        r"(\d{4}-\d{2}-\d{2})[T_ ](\d{2})[-_](\d{2})[-_](\d{2})",
        r"(\d{4}-\d{2}-\d{2})[T_ ](\d{2})_(\d{2})_(\d{2})",
        # inside filenames like behavior2024-05-24T18_42_16
        r"(\d{4}-\d{2}-\d{2})T(\d{2})_(\d{2})_(\d{2})",
        # fallback date only
        r"(\d{4}-\d{2}-\d{2})"
    ]

    for pat in patterns:
        m = re.search(pat, path)
        if m:
            groups = m.groups()
            # if only date matched
            if len(groups) == 1:
                return groups[0]
            # format safely
            date = groups[0]
            time = "-".join(groups[1:4])
            return f"{date}_{time}"

    return "unknown_datetime"



def unit(v: np.ndarray) -> np.ndarray:
    v = np.asarray(v, float)
    n = np.linalg.norm(v)
    if n == 0:
        return v * 0.0
    return v / n


def pick_most_robust_bodypart(dlc_df: pd.DataFrame, thr: float = 0.9) -> str:
    """
    Return bodypart name with highest fraction of frames having likelihood > thr.

    Assumes DLC columns are a 3-level MultiIndex: (scorer, bodypart, coord)
    where coord in {'x', 'y', 'likelihood'}.
    """
    if not isinstance(dlc_df.columns, pd.MultiIndex):
        raise ValueError("DLC DataFrame columns must be a MultiIndex (scorer, bodypart, coord).")

    lcols = [
        c for c in dlc_df.columns
        if len(c) == 3 and str(c[2]).strip().lower() == "likelihood"
    ]
    if not lcols:
        raise ValueError("No likelihood columns found in DLC data.")
    scores = {}
    for col in lcols:
        bp = col[1]
        s = pd.to_numeric(dlc_df[col], errors="coerce")
        total = s.notna().sum()
        if total == 0:
            continue
        score = (s > thr).sum() / total
        scores[bp] = max(scores.get(bp, 0.0), float(score))
    if not scores:
        raise ValueError("No valid likelihood data to compute robustness scores.")
    best_bp = max(scores.items(), key=lambda kv: kv[1])[0]
    return best_bp


def col_of(dlc_df: pd.DataFrame, bodypart: str, coord: str):
    """
    Return the MultiIndex column for a given bodypart and coord ('x', 'y', 'likelihood').
    """
    coord = coord.lower()
    for col in dlc_df.columns:
        if len(col) == 3 and str(col[1]) == bodypart and str(col[2]).lower() == coord:
            return col
    raise KeyError(f"Could not find coord={coord!r} for bodypart={bodypart!r} in DLC columns.")


def load_trim_info(trim_csv_path: str) -> pd.Series:
    """
    Load dlc_trim_info.csv and return its first row.

    Expected columns:
      - 'start_frame'  (required)
      - and optionally something like 'bodypart', 'best_bodypart', etc.

    Note: dlc_trim_info.csv (as produced by trim_videos_from_dlc.py) never
    contains an 'end_frame' column -- the end of the trim window is derived
    from the session's neural (CaImAn) frame count instead, see
    process_lineartrack_session().
    """
    trim_df = pd.read_csv(trim_csv_path)
    if trim_df.empty:
        raise ValueError(f"{trim_csv_path} is empty.")
    if "start_frame" not in trim_df.columns:
        raise ValueError("dlc_trim_info.csv must contain a 'start_frame' column.")
    return trim_df.iloc[0]


def choose_bodypart_from_trim_info(
    trim_row: pd.Series,
    dlc_df: pd.DataFrame,
    fallback_thr: float = 0.9,
) -> str:
    """
    Try to read bodypart name from trim_row; if missing, fall back to
    pick_most_robust_bodypart on dlc_df.
    """
    for key in ["bodypart", "best_bodypart", "robust_bodypart", "most_robust_bodypart"]:
        if key in trim_row.index and isinstance(trim_row[key], str) and trim_row[key]:
            return trim_row[key]
    # Fallback
    return pick_most_robust_bodypart(dlc_df, thr=fallback_thr)


def load_dlc_for_bodypart(
    dlc_csv_path: str,
    bodypart: str,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, pd.DataFrame]:
    """
    Load DLC CSV and return (x, y, likelihood, full_df) for a single bodypart.
    """
    dlc_df = pd.read_csv(dlc_csv_path, header=[0, 1, 2])

    x_col = col_of(dlc_df, bodypart, "x")
    y_col = col_of(dlc_df, bodypart, "y")
    l_col = col_of(dlc_df, bodypart, "likelihood")

    x = pd.to_numeric(dlc_df[x_col], errors="coerce").to_numpy()
    y = pd.to_numeric(dlc_df[y_col], errors="coerce").to_numpy()
    p = pd.to_numeric(dlc_df[l_col], errors="coerce").to_numpy()

    return x, y, p, dlc_df

# =========================
# Neural length helper
# =========================

def load_neural_length(caiman_path: str) -> int:
    """
    Load neural activity length (number of frames).

    Tries CaImAn's cnmf.load_CNMF first; if that fails, falls back to reading
    '/estimates/F_dff' from the HDF5 file.
    """
    try:
        from caiman.source_extraction.cnmf.cnmf import load_CNMF  # type: ignore
        cnmf_obj = load_CNMF(caiman_path)
        F_dff = cnmf_obj.estimates.F_dff[cnmf_obj.estimates.idx_components]
        return int(F_dff.shape[1])
    except Exception:
        pass

    with h5py.File(caiman_path, "r") as f:
        if "estimates" in f and "F_dff" in f["estimates"]:
            dset = f["estimates"]["F_dff"]
            return int(dset.shape[1])
        raise RuntimeError(
            f"Could not determine neural length from {caiman_path}. "
            "Install CaImAn or adjust load_neural_length."
        )


def find_caiman_results(session_dir: str) -> str:
    """
    Look for caiman_results.hdf5 under the standard caiman output folders.

    caiman_aging/        -> aging cohort, current best-parameter reprocessing (preferred
                             whenever present -- supersedes any caiman_final/ for the same
                             session)
    caiman_final/        -> young cohort, and any aging-cohort session not yet reprocessed
                             with aging-specific gSig/gSig_filt
    caiman_final_young/  -> legacy naming, kept as a fallback (only 2 known
                             sessions still use it: Mouse943 Round1 Linear
                             2025-11-17T17_15_35 and 2025-11-18T17_44_50)
    """
    for subdir in ("caiman_aging", "caiman_final", "caiman_final_young"):
        guess = os.path.join(session_dir, subdir, "caiman_results.hdf5")
        if os.path.isfile(guess):
            return guess
    raise FileNotFoundError(
        f"caiman_results.hdf5 not found under {session_dir}/caiman_aging/, "
        f"{session_dir}/caiman_final/, or {session_dir}/caiman_final_young/; "
        "please pass caiman_path explicitly."
    )

# =========================
# Trajectory cleaning
# =========================

def process_location_2D(
    location: np.ndarray,
    likelihood: np.ndarray,
    likelihood_threshold: float = LIKELIHOOD_THRESHOLD,
    max_jump: float = MAX_JUMP,
    smoothing_sigma: float = SMOOTHING_SIGMA,
) -> np.ndarray:
    """
    Basic trajectory cleaning:

    1) Remove samples with low likelihood.
    2) Remove large frame-to-frame jumps (> max_jump) by marking them as bad.
    3) Linearly interpolate over bad samples.
    4) Gaussian smooth each coordinate.
    """
    loc = np.asarray(location, float)
    lh = np.asarray(likelihood, float).squeeze()

    if loc.ndim != 2 or loc.shape[1] != 2:
        raise ValueError("location must be (N, 2) array")
    if loc.shape[0] != lh.shape[0]:
        raise ValueError("location and likelihood must have same length")

    n = loc.shape[0]

    # 1) Mark low-likelihood samples as bad
    bad = ~(lh >= likelihood_threshold)

    # 2) Mark large jumps as bad
    dx = np.diff(loc[:, 0])
    dy = np.diff(loc[:, 1])
    jump = np.hypot(dx, dy)
    jump = np.concatenate([[0.0], jump])  # same length as loc
    bad |= jump > max_jump

    # 3) Interpolate each coordinate separately over bad samples
    cleaned = loc.copy()
    for dim in range(2):
        arr = cleaned[:, dim]
        good_idx = np.where(~bad & ~np.isnan(arr))[0]
        bad_idx = np.where(bad | np.isnan(arr))[0]
        if len(good_idx) == 0:
            continue
        if len(bad_idx) > 0:
            arr[bad_idx] = np.interp(bad_idx, good_idx, arr[good_idx])
        cleaned[:, dim] = arr

    # 4) Smooth
    for dim in range(2):
        cleaned[:, dim] = gaussian_filter1d(cleaned[:, dim], sigma=smoothing_sigma)

    return cleaned


def get_movement_mask_2D(
    location: np.ndarray,
    speed_threshold: float = SPEED_THRESHOLD,
    min_speed_duration: float = MIN_SPEED_DURATION,
    samples_per_second: float = FPS,
) -> Tuple[np.ndarray, List[Tuple[int, int]]]:
    """
    Compute a boolean mask of "movement" frames from 2D coordinates.
    """
    loc = np.asarray(location, float)
    if loc.ndim != 2 or loc.shape[1] != 2:
        raise ValueError("location must be (N, 2) array")

    vx = np.gradient(loc[:, 0])
    vy = np.gradient(loc[:, 1])
    speed = np.hypot(vx, vy)

    speed_bool = speed >= speed_threshold

    periods: List[Tuple[int, int]] = []
    in_run = False
    start = 0
    for i, val in enumerate(speed_bool):
        if val and not in_run:
            in_run = True
            start = i
        elif not val and in_run:
            in_run = False
            periods.append((start, i - 1))
    if in_run:
        periods.append((start, len(speed_bool) - 1))

    # Keep only sufficiently long periods
    min_samples = int(np.round(min_speed_duration * samples_per_second))
    filtered_periods = [
        (a, b) for (a, b) in periods if (b - a + 1) >= min_samples
    ]

    # Build mask
    mask = np.zeros(loc.shape[0], dtype=bool)
    for a, b in filtered_periods:
        mask[a:b + 1] = True

    return mask, filtered_periods


def apply_bounding_boxes(
    location: np.ndarray,
    boxes: Optional[Sequence[Tuple[float, float, float, float]]] = None,
) -> np.ndarray:
    """
    Optionally zero-out points outside user-defined bounding boxes.

    boxes is a sequence of (xmin, xmax, ymin, ymax). A point is kept if it is inside
    *any* box. If boxes is None or empty, the input is returned unchanged.
    """
    loc = np.asarray(location, float)
    if boxes is None or len(list(boxes)) == 0:
        return loc

    mask_any = np.zeros(loc.shape[0], dtype=bool)
    for xmin, xmax, ymin, ymax in boxes:
        inside = (
            (loc[:, 0] >= xmin) & (loc[:, 0] <= xmax) &
            (loc[:, 1] >= ymin) & (loc[:, 1] <= ymax)
        )
        mask_any |= inside

    loc[~mask_any, :] = np.nan
    return loc

# =========================
# Ports, trial segmentation
# =========================

def roi_center(roi: Tuple[float, float, float, float]) -> Tuple[float, float]:
    """Center point of a (xmin, xmax, ymin, ymax) rectangular ROI."""
    xmin, xmax, ymin, ymax = roi
    return (xmin + xmax) / 2.0, (ymin + ymax) / 2.0


def inside_port(xy: Sequence[float], name: str, port_rois: Dict[str, Tuple[float, float, float, float]]) -> bool:
    xmin, xmax, ymin, ymax = port_rois[name]
    return xmin <= xy[0] <= xmax and ymin <= xy[1] <= ymax


def label_ports_generic(
    XY: np.ndarray,
    valid_mask: np.ndarray,
    port_rois: Dict[str, Tuple[float, float, float, float]],
    min_dwell_frames: int = MIN_DWELL_FRAMES,
) -> np.ndarray:
    """
    Return array of port labels (any keys in port_rois, or None) for each frame.
    Uses a dwell counter to avoid flickering labels.
    """
    labels = np.empty(len(XY), dtype=object)
    current = None
    dwell = 0
    port_names = list(port_rois.keys())

    for i, xy in enumerate(XY):
        if not valid_mask[i] or np.any(np.isnan(xy)):
            labels[i] = current
            continue

        hit = None
        # Pick the first port whose ROI contains the point
        for nm in port_names:
            if inside_port(xy, nm, port_rois):
                hit = nm
                break

        if hit == current:
            if hit is not None:
                dwell += 1
        else:
            if hit is None:
                current = None
                dwell = 0
            else:
                dwell += 1
                if dwell >= min_dwell_frames:
                    current = hit
        labels[i] = current

    return labels


def classify_trial_generic(origin: str, dest: str) -> str:
    if origin is None or dest is None or origin == dest:
        return "other"
    return f"{origin}_to_{dest}"


def build_dir_templates_generic(
    port_rois: Dict[str, Tuple[float, float, float, float]]
) -> Dict[Tuple[str, str], np.ndarray]:
    """
    Build direction templates for every ordered pair of distinct ports.
    """
    names = list(port_rois.keys())
    templates: Dict[Tuple[str, str], np.ndarray] = {}
    for i, origin in enumerate(names):
        for j, dest in enumerate(names):
            if i == j:
                continue
            ox, oy = roi_center(port_rois[origin])
            dx, dy = roi_center(port_rois[dest])
            templates[(origin, dest)] = unit(np.array([dx - ox, dy - oy], float))
    return templates


def segment_trials_from_trajectory(
    XY: np.ndarray,
    likelihood: np.ndarray,
    port_rois: Dict[str, Tuple[float, float, float, float]],
    fps: float = FPS,
    min_trial_duration_s: float = MIN_TRIAL_DURATION_S,
    min_dir_cos: float = MIN_DIR_COS,
    speed_min: float = 0.0,
) -> pd.DataFrame:
    """
    Trial segmentation logic adapted to work on an already-trimmed trajectory.

    Returns a DataFrame with (start_frame, end_frame, origin, destination,
    duration_s, mean_speed, dir_score, trial_type) in *local* frame indices.

    A trial starts when the animal leaves a port, and ends when it enters any
    (other) port. Frames *inside* ports are not included in trial trajectories.
    """
    # Compute speed (after light median filter to reduce noise)
    x_s = XY[:, 0]
    y_s = XY[:, 1]
    x_med = medfilt(x_s, kernel_size=5)
    y_med = medfilt(y_s, kernel_size=5)

    vx = np.gradient(x_med) * fps
    vy = np.gradient(y_med) * fps
    speed_raw = np.hypot(vx, vy)

    # Likelihood / movement masks
    good_lik = likelihood >= LIKELIHOOD_THRESHOLD
    moving_raw = speed_raw >= speed_min
    valid = good_lik & moving_raw

    # Port labels and direction templates
    port_label = label_ports_generic(XY, good_lik, port_rois)
    dir_templates = build_dir_templates_generic(port_rois)
    min_trial_len = int(min_trial_duration_s * fps)

    trials: List[Dict[str, object]] = []
    state_port = port_label[0]
    in_transit = False
    origin = None
    start_idx: Optional[int] = None

    for i in range(1, len(port_label)):
        cur = port_label[i]
        if not in_transit:
            # Start of trial: leaving a port
            if state_port is not None and cur is None:
                in_transit = True
                origin = state_port
                start_idx = i - 1
        else:
            # End of trial: entering a port
            if cur is not None:
                dest = cur
                end_idx = i

                if (
                    origin is not None
                    and dest is not None
                    and origin != dest
                    and (origin, dest) in dir_templates
                ):
                    seg = slice(start_idx, end_idx + 1)
                    seg_valid = valid[seg] & (speed_raw[seg] > speed_min)

                    if seg_valid.sum() >= 3 and (end_idx - start_idx + 1) >= min_trial_len:
                        vx_seg = vx[seg][seg_valid]
                        vy_seg = vy[seg][seg_valid]
                        sp_seg = speed_raw[seg][seg_valid]

                        templ = dir_templates[(origin, dest)]
                        vvec = np.column_stack([vx_seg, vy_seg])
                        cos_vals = (vvec @ templ) / (np.linalg.norm(vvec, axis=1) + 1e-9)
                        mean_cos = float(np.nanmean(cos_vals))

                        if mean_cos >= min_dir_cos:
                            trials.append(
                                {
                                    "start_frame": int(start_idx),
                                    "end_frame": int(end_idx),
                                    "origin": origin,
                                    "destination": dest,
                                    "duration_s": (end_idx - start_idx + 1) / fps,
                                    "mean_speed": float(np.nanmean(sp_seg)),
                                    "dir_score": mean_cos,
                                    "trial_type": classify_trial_generic(origin, dest),
                                }
                            )

                in_transit = False
                origin = None
                start_idx = None

        state_port = cur

    # Always return the expected columns, even with zero trials -- an empty
    # pd.DataFrame(trials) would otherwise have no columns at all, breaking every
    # downstream trials_df['start_frame'] access for sessions with no valid trials.
    return pd.DataFrame(trials, columns=["start_frame", "end_frame", "origin", "destination", "duration_s", "mean_speed", "dir_score", "trial_type"])

# =========================
# Linearization & binning
# =========================

def compute_track_axis(port_rois: Dict[str, Tuple[float, float, float, float]]) -> Tuple[np.ndarray, np.ndarray, float]:
    """
    For a two-port linear track, build the main axis from 'start' to 'end'.

    Returns:
      origin   : 2D point (start port center)
      axis_vec : unit vector along track (start → end)
      length   : scalar track length (distance between port centers)
    """
    if "left" not in port_rois or "right" not in port_rois:
        raise ValueError("For linear track, port_rois must have 'left' and 'right' keys.")
    sx, sy = roi_center(port_rois["left"])
    ex, ey = roi_center(port_rois["right"])
    origin = np.array([sx, sy], float)
    vec = np.array([ex - sx, ey - sy], float)
    length = float(np.linalg.norm(vec))
    if length == 0:
        raise ValueError("Left and right ports have identical coordinates.")
    axis_vec = vec / length
    return origin, axis_vec, length


def linearize_positions(
    XY: np.ndarray,
    port_rois: Dict[str, Tuple[float, float, float, float]],
) -> Tuple[np.ndarray, np.ndarray, float]:
    """
    Project XY points onto track axis defined by 'start' and 'end' ports.

    Returns:
      s         : 1D positions along track, in pixel units from 'start'
      s_norm    : normalized positions in [0, 1] (clipped)
      track_len : total track length in pixels
    """
    origin, axis_vec, track_len = compute_track_axis(port_rois)
    rel = XY - origin[None, :]
    s = rel @ axis_vec
    s_norm = s / track_len
    s_norm = np.clip(s_norm, 0.0, 1.0)
    return s, s_norm, track_len


def bin_positions(
    s: np.ndarray,
    n_bins: int,
    port_labels: np.ndarray,
    track_len: float,
) -> np.ndarray:
    """
    Assign each position to a spatial bin along the track.

    Frames that are NaN, outside [0, track_len], or inside a port (port_labels != None)
    get bin = -1.
    """
    s = np.asarray(s, float)
    bins = np.full_like(s, fill_value=-1, dtype=int)

    good = ~np.isnan(s) & (s >= 0.0) & (s <= track_len) & (pd.isna(port_labels) | (port_labels == None))  # noqa: E711
    if not np.any(good):
        return bins

    edges = np.linspace(0.0, track_len, n_bins + 1)
    idx = np.digitize(s[good], edges, right=False) - 1
    idx = np.clip(idx, 0, n_bins - 1)
    bins[good] = idx
    return bins

# =========================
# Main pipeline
# =========================

def process_lineartrack_session(
    session_dir: str,
    trim_info_name: str = "dlc_trim_info.csv",
    dlc_pattern: str = "*_filtered.csv",
    caiman_path: Optional[str] = None,
    port_rois: Optional[Dict[str, Tuple[float, float, float, float]]] = None,
    bounding_boxes: Optional[Sequence[Tuple[float, float, float, float]]] = None,
    output_dir: Optional[str] = None,
    n_bins: int = N_BINS,
) -> None:
    """
    Run the full linear-track pipeline for a single session.

    Expected layout:
      session_dir/
        dlc_trim_info.csv
        dlc/
          <one DLC CSV matching dlc_pattern>
        caiman_final/caiman_results.hdf5  (or caiman_final_young/)

    Steps:
      1) Read trim info, DLC, and neural length (from caiman_results.hdf5)
      2) Determine bodypart
      3) Trim to [start_frame, start_frame + neural_length) -- bounds the
         trajectory to exactly the window CaImAn actually processed, the
         same way TMaze_combined_pipeline.ipynb's process_session() does.
         (dlc_trim_info.csv never has an end_frame column, so without this
         the trajectory silently ran to the end of the full, untrimmed DLC
         file instead of the ~960s window that was actually analyzed.)
      4) Clean 2D trajectory, then apply bounding boxes (if any)
      5) Detect movement (optional)
      6) Label ports (start / end) and segment trials
      7) Linearize and bin positions
      8) Save CSVs
    """
    session_dir = os.path.abspath(session_dir)

    # 1) Paths
    trim_info_path = os.path.join(session_dir, trim_info_name)
    dlc_dir = os.path.join(session_dir, "dlc")
    dlc_files = glob.glob(os.path.join(dlc_dir, dlc_pattern))
    if not dlc_files:
        raise FileNotFoundError(f"No DLC files matching {dlc_pattern!r} in {dlc_dir}")
    # Exclude the home-box camera's DLC output -- for the aging cohort (2024),
    # "behaviorLinear<TS>..." is the home-box video, the opposite of what its name suggests
    # (see CLAUDE.md's naming-inversion warning). A dlc/ folder can also carry outputs from
    # older DLC snapshots (e.g. 100000) copied alongside the current one -- always prefer
    # snapshot 160000 per CLAUDE.md's DLC re-processing policy.
    arena_files = [f for f in dlc_files if not os.path.basename(f).startswith("behaviorLinear")]
    preferred = [f for f in arena_files if "_160000_filtered.csv" in os.path.basename(f)]
    if not preferred:
        raise FileNotFoundError(
            f"No snapshot-160000 DLC file found in {dlc_dir} (per CLAUDE.md, always prefer "
            f"snapshot 160000). Candidates found: {[os.path.basename(f) for f in arena_files]}. "
            "This session needs DLC re-run with the 160000 snapshot -- see CLAUDE.md's "
            "'Pending DLC Re-processing' list."
        )
    if len(preferred) > 1:
        raise RuntimeError(
            f"More than one snapshot-160000 DLC file found in {dlc_dir}, please narrow "
            f"dlc_pattern: {preferred}"
        )
    dlc_csv_path = preferred[0]

    if port_rois is None:
        port_rois = PORT_ROIS_DEFAULT

    if output_dir is None:
        output_dir = dlc_dir
    os.makedirs(output_dir, exist_ok=True)

    # 2) Neural length (bounds the trim window -- see docstring)
    if caiman_path is None:
        caiman_path = find_caiman_results(session_dir)
    neural_len = load_neural_length(caiman_path)
    print(f"[{session_dir}] neural_length = {neural_len} frames")

    # 3) DLC + trim info
    trim_row = load_trim_info(trim_info_path)
    # Align to where the clip ACTUALLY starts, not where the trim was asked to start.
    # trim_videos_from_dlc.py cuts with `ffmpeg -ss <t> -i raw ... -c copy`, and input
    # seeking with stream copy cannot begin mid-GOP: ffmpeg backs off to an earlier
    # keyframe, so the clip starts k frames BEFORE `start_frame` (k measured per session
    # by scripts/data_audit/measure_trim_offset.py; 0-57 frames observed). Slicing the DLC
    # table at `start_frame` pairs pose row i with a neural frame k frames earlier, i.e.
    # behaviour runs AHEAD of the neural data by up to ~2 s, the same direction every
    # session. Fall back to `start_frame` for sessions not yet measured.
    if "start_frame_actual" in trim_row.index and pd.notna(trim_row["start_frame_actual"]):
        trim_start_frame = int(trim_row["start_frame_actual"])
    else:
        trim_start_frame = int(trim_row["start_frame"])

    # load DLC once to resolve bodypart and coordinates
    dlc_df_full = pd.read_csv(dlc_csv_path, header=[0, 1, 2])
    bodypart = choose_bodypart_from_trim_info(trim_row, dlc_df_full, fallback_thr=LIKELIHOOD_THRESHOLD)
    print(f"[{session_dir}] using bodypart = {bodypart!r}")

    x_full = pd.to_numeric(dlc_df_full[col_of(dlc_df_full, bodypart, "x")], errors="coerce").to_numpy()
    y_full = pd.to_numeric(dlc_df_full[col_of(dlc_df_full, bodypart, "y")], errors="coerce").to_numpy()
    p_full = pd.to_numeric(dlc_df_full[col_of(dlc_df_full, bodypart, "likelihood")], errors="coerce").to_numpy()

    n_frames_full = len(x_full)
    trim_end_frame = min(trim_start_frame + neural_len, n_frames_full)
    if trim_end_frame <= trim_start_frame:
        raise ValueError("Trim window is empty; check start_frame and neural length.")

    # 4) Trim to recording window
    sl = slice(trim_start_frame, trim_end_frame)
    x_trim = x_full[sl]
    y_trim = y_full[sl]
    p_trim = p_full[sl]

    local_frames = np.arange(len(x_trim), dtype=int)
    global_frames = local_frames + trim_start_frame
    XY_raw = np.column_stack([x_trim, y_trim])

    # 5) Clean, then apply bounding boxes (optional).
    # Order matters: cleaning interpolates over NaNs, so boxing-then-cleaning
    # (the old order) would silently interpolate right back over any point a
    # box excluded, making bounding_boxes a no-op. Clean first (matches
    # TMaze_combined_pipeline.ipynb's process_session order) so a box's NaNs
    # actually persist into XY_clean.
    XY_clean = process_location_2D(XY_raw, p_trim, likelihood_threshold=LIKELIHOOD_THRESHOLD)
    XY_clean = apply_bounding_boxes(XY_clean, bounding_boxes)

    # 6) Movement mask
    movement_mask, _ = get_movement_mask_2D(XY_clean)

    # 7) Port labels and trials
    port_labels = label_ports_generic(XY_clean, p_trim >= LIKELIHOOD_THRESHOLD, port_rois=port_rois)
    trials_df = segment_trials_from_trajectory(
        XY_clean,
        p_trim,
        port_rois=port_rois,
        fps=FPS,
        min_trial_duration_s=MIN_TRIAL_DURATION_S,
        min_dir_cos=MIN_DIR_COS,
        speed_min=0.0,
    )
    trials_df["start_frame_global"] = trials_df["start_frame"] + trim_start_frame
    trials_df["end_frame_global"] = trials_df["end_frame"] + trim_start_frame

    # 8) Linearize and bin positions
    s, s_norm, track_len = linearize_positions(XY_clean, port_rois=port_rois)
    spatial_bin = bin_positions(s, n_bins=n_bins, port_labels=port_labels, track_len=track_len)

    # 9) Save outputs
    stamp = extract_datetime_stamp(dlc_csv_path)
    prefix = f"{stamp}_LinearTrack"

    # 8a) Frame-wise raw & cleaned positions, ports, linearized coord, bins
    df_frames = pd.DataFrame(
        {
            "frame_local": local_frames,
            "frame_global": global_frames,
            "x_raw": x_trim,
            "y_raw": y_trim,
            "likelihood": p_trim,
            "x_clean": XY_clean[:, 0],
            "y_clean": XY_clean[:, 1],
            "moving": movement_mask.astype(int),
            "port_label": port_labels,
            "s_linear": s,
            "s_norm": s_norm,
            "spatial_bin": spatial_bin,
        }
    )
    frames_path = os.path.join(output_dir, f"{prefix}_lineartrack_positions_ports_bins_{n_bins}nbins.csv")
    df_frames.to_csv(frames_path, index=False)

    # 8b) Trials table (local and global indices)
    trials_path = os.path.join(output_dir, f"{prefix}_lineartrack_trials.csv")
    trials_df.to_csv(trials_path, index=False)

    # 8c) Trial-by-trial cleaned, linearized, and binned locations (long format)
    rows_clean = []
    for trial_id, tr in trials_df.iterrows():
        seg = slice(int(tr["start_frame"]), int(tr["end_frame"]) + 1)
        for local_idx, frame_idx in enumerate(range(seg.start, seg.stop)):
            fl = frame_idx
            fg = fl + trim_start_frame
            rows_clean.append(
                {
                    "trial_id": int(trial_id),
                    "trial_type": tr["trial_type"],
                    "origin": tr["origin"],
                    "destination": tr["destination"],
                    "frame_in_trial": int(local_idx),
                    "frame_local": int(fl),
                    "frame_global": int(fg),
                    "x_clean": float(XY_clean[fl, 0]),
                    "y_clean": float(XY_clean[fl, 1]),
                    "linear_pos": float(s[fl]),
                    "s_norm": float(s_norm[fl]),
                    "spatial_bin": int(spatial_bin[fl]),
                }
            )

    df_trials_clean = pd.DataFrame(rows_clean)
    trials_clean_path = os.path.join(
        output_dir,
        f"{prefix}_linearized_binned_trials_{n_bins}nbins.csv",
    )
    df_trials_clean.to_csv(trials_clean_path, index=False)

    print("Saved:")
    print("  ", frames_path)
    print("  ", trials_path)
    print("  ", trials_clean_path)
    print("Done.")


In [ ]:
# session_dir = "/media/toor/T7Shield/Mouse1637/Round1/Linear/2024-06-06T14_35_51"
# process_lineartrack_session(session_dir, port_rois=PORT_ROIS_DEFAULT, n_bins=70)

### Batch process

In [ ]:
from pathlib import Path
ROOT_DIR = Path("/media/toor/SeagateClean/Mouse1639/Round1.5/Linear")  # <-- EDIT

SESSION_DIRS = [Path(subdir) for subdir in ROOT_DIR.iterdir() if subdir.is_dir() and "Registration" not in subdir.name] 

In [ ]:
for sess_dir in SESSION_DIRS:
    print(f"Processing session: {sess_dir}")
    port_rois = PORT_ROIS_BY_ERA[era_for_session(sess_dir)]
    process_lineartrack_session(sess_dir, port_rois=port_rois, n_bins=70)

### Sanity Check Plots

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap


def load_lineartrack_outputs(
    frames_csv_path: str,
    trials_csv_path: str,
):
    """
    Load the frame-wise and trial-wise CSVs produced by process_lineartrack_session.
    Returns (df_frames, df_trials).
    """
    df_frames = pd.read_csv(frames_csv_path)
    df_trials = pd.read_csv(trials_csv_path)
    
    # Sanity: make sure required columns exist
    required_frame_cols = {"frame_local", "x_clean", "y_clean"}
    required_trial_cols = {"start_frame", "end_frame", "trial_type"}
    if not required_frame_cols.issubset(df_frames.columns):
        raise ValueError(f"frames CSV missing required columns: {required_frame_cols - set(df_frames.columns)}")
    if not required_trial_cols.issubset(df_trials.columns):
        raise ValueError(f"trials CSV missing required columns: {required_trial_cols - set(df_trials.columns)}")
    
    return df_frames, df_trials


def plot_trials_xy(
    df_frames: pd.DataFrame,
    df_trials: pd.DataFrame,
    max_trials_per_type: int = 20,
    subsample: int = 1,
    title: str = "XY trajectory per trial",
):
    """
    Quick sanity-check plot:
      - For each trial, plot x_clean vs y_clean as a line
      - Color by trial_type
      - Optionally limit number of trials per type and subsample frames

    Assumes:
      - df_frames has columns: frame_local, x_clean, y_clean
      - df_trials has columns: start_frame, end_frame, trial_type
        where start_frame and end_frame are in LOCAL frame indices (0-based)
    """
    # Build color map for trial types
    trial_types = sorted(df_trials["trial_type"].dropna().unique())
    cmap = get_cmap("tab10")
    type_to_color = {tt: cmap(i % 10) for i, tt in enumerate(trial_types)}

    fig, ax = plt.subplots(figsize=(6, 6))

    for tt in trial_types:
        df_tt = df_trials[df_trials["trial_type"] == tt].reset_index(drop=True)
        n_show = min(len(df_tt), max_trials_per_type)
        
        for tr_id in range(n_show):
            tr = df_tt.iloc[tr_id]
            start = int(tr["start_frame"])
            end = int(tr["end_frame"])
            seg_idx = np.arange(start, end + 1, subsample)

            # Note: df_frames index is assumed to be 0..N-1 matching frame_local
            xy = df_frames.loc[seg_idx, ["x_clean", "y_clean"]].to_numpy()
            
            ax.plot(
                xy[:, 0],
                xy[:, 1],
                color=type_to_color[tt],
                alpha=0.4,
                linewidth=1,
            )

    # Make it look like the video
    ax.set_aspect("equal", adjustable="box")
    ax.invert_yaxis()
    ax.set_xlabel("x_clean (pixels)")
    ax.set_ylabel("y_clean (pixels)")
    ax.set_title(title)

    # Legend handles
    handles = [
        plt.Line2D([0], [0], color=type_to_color[tt], lw=2, label=tt)
        for tt in trial_types
    ]
    ax.legend(handles=handles, title="trial_type", loc="best")

    plt.show()


In [ ]:
# Example paths for one session
ROOT_DIR = Path("/media/toor/SeagateClean/Mouse945/Round1/Linear")  # <-- EDIT

frames_csv_path = f"{ROOT_DIR}/2025-11-25T18_05_49/dlc/2025-11-25_18-05-49_LinearTrack_lineartrack_positions_ports_bins_70nbins.csv"
trials_csv_path = "/media/toor/Seagate2/Mouse943/Round1/Linear/2025-11-25T18_05_49/dlc/2025-11-25_18-05-49_LinearTrack_lineartrack_trials.csv"



df_frames, df_trials = load_lineartrack_outputs(frames_csv_path, trials_csv_path)

print("Frames:", df_frames.shape)
print("Trials:", df_trials.shape)
print(df_trials.head())

plot_trials_xy(
    df_frames,
    df_trials,
    max_trials_per_type=30,      # limit per trial_type to avoid spaghetti
    subsample=2,                 # take every 2nd frame for lighter plots
    title="Linear track XY per trial (sanity check)",
)


In [ ]:
# --------------------------------------------------
# 1. Get first session folder inside ROOT_DIR
# --------------------------------------------------
session_folder = sorted([p for p in ROOT_DIR.iterdir() if p.is_dir()])[0]

# --------------------------------------------------
# 2. Go inside dlc subfolder
# --------------------------------------------------
dlc_folder = session_folder / "dlc"

# --------------------------------------------------
# 3. Grab first CSV ending with 'bins.csv'
# --------------------------------------------------
frames_csv_path = sorted(dlc_folder.glob("*ports*bins.csv"))[0]

# --------------------------------------------------
# 4. Grab first CSV ending with '_trials.csv'
# --------------------------------------------------
trials_csv_path = sorted(dlc_folder.glob("*_trials.csv"))[0]

print("Frames CSV:", frames_csv_path)
print("Trials CSV:", trials_csv_path)

# --------------------------------------------------
# Load
# --------------------------------------------------
df_frames, df_trials = load_lineartrack_outputs(
    str(frames_csv_path),
    str(trials_csv_path)
)

print("Frames:", df_frames.shape)
print("Trials:", df_trials.shape)
print(df_trials.head())

plot_trials_xy(
    df_frames,
    df_trials,
    max_trials_per_type=30,
    subsample=2,
    title="Linear track XY per trial (sanity check)",
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_trials_xy_grid(
    df_frames,
    df_trials,
    max_trials=20,
    ncols=5,
    subsample=1,
    title_prefix="XY, trial",
):
    """
    Make a small panel for each trial: x_clean vs y_clean.
    """
    # pick which trials to show (first max_trials)
    trials_to_plot = df_trials.reset_index(drop=True).iloc[:max_trials]

    ntr = len(trials_to_plot)
    ncols = min(ncols, ntr)
    nrows = int(np.ceil(ntr / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(3*ncols, 3*nrows), squeeze=False)

    for k, (_, tr) in enumerate(trials_to_plot.iterrows()):
        r = k // ncols
        c = k % ncols
        ax = axes[r, c]

        start = int(tr["start_frame"])
        end   = int(tr["end_frame"])
        seg_idx = np.arange(start, end + 1, subsample)

        xy = df_frames.loc[seg_idx, ["x_clean", "y_clean"]].to_numpy()

        ax.plot(xy[:, 0], xy[:, 1], lw=1)

        ax.set_aspect("equal", adjustable="box")
        ax.invert_yaxis()
        ax.set_xticks([])
        ax.set_yticks([])

        tt = tr.get("trial_type", "NA")
        ax.set_title(f"{title_prefix} {k} ({tt})", fontsize=8)

    # hide any extra axes
    for k in range(ntr, nrows*ncols):
        r = k // ncols
        c = k % ncols
        axes[r, c].axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
# plot_trials_xy_grid(df_frames, df_trials,
#                     max_trials=24,   # how many panels total
#                     ncols=6,         # panels per row
#                     subsample=1)


In [ ]:
def plot_trials_bins_grid(
    df_frames,
    df_trials,
    max_trials=20,
    ncols=5,
    title_prefix="Bins, trial",
):
    """
    Panel per trial showing spatial_bin as a function of frame_in_trial.
    Negative bins (-1) are plotted as gaps.
    """
    if "spatial_bin" not in df_frames.columns:
        raise ValueError("df_frames must have a 'spatial_bin' column")

    trials_to_plot = df_trials.reset_index(drop=True).iloc[:max_trials]

    ntr = len(trials_to_plot)
    ncols = min(ncols, ntr)
    nrows = int(np.ceil(ntr / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(3*ncols, 3*nrows), squeeze=False)

    for k, (_, tr) in enumerate(trials_to_plot.iterrows()):
        r = k // ncols
        c = k % ncols
        ax = axes[r, c]

        start = int(tr["start_frame"])
        end   = int(tr["end_frame"])
        idx = np.arange(start, end + 1)

        bins = df_frames.loc[idx, "spatial_bin"].to_numpy().astype(float)
        # convert invalid bins to nan so they show as gaps
        bins[bins < 0] = np.nan

        frame_in_trial = np.arange(len(idx))

        ax.plot(frame_in_trial, bins, lw=1)

        ax.set_xlabel("frame in trial", fontsize=7)
        ax.set_ylabel("bin", fontsize=7)
        ax.tick_params(labelsize=6)

        tt = tr.get("trial_type", "NA")
        ax.set_title(f"{title_prefix} {k} ({tt})", fontsize=8)

    # hide extras
    for k in range(ntr, nrows*ncols):
        r = k // ncols
        c = k % ncols
        axes[r, c].axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
plot_trials_bins_grid(df_frames, df_trials,
                      max_trials=24,
                      ncols=6)


### Decide on port ROI locations - ONLY NEEDED TO DO THIS ONCE 

In [ ]:
import pandas as pd
import numpy as np

# 1. Path to your DLC CSV -- one option per camera era (see PORT_ROIS_BY_ERA /
# era_for_session above), plus the aging cohort, which shares pre2026-05's
# camera position. Switch DLC_ERA to check a different era's calibration.
DLC_PATH_OPTIONS = {
    "aging2024":   "/media/toor/SeagateClean/Mouse1639/Round1/Linear/2024-05-23T17_10_09/dlc/behavior2024-05-23T17_10_09DLC_resnet50_linearMay29shuffle1_160000_filtered.csv",
    "pre2026-05":  "/media/toor/SeagateClean/Mouse945/Round1/Linear/2026-01-21T17_26_52/dlc/Linear2026-01-21T17_26_52DLC_resnet50_linearMay29shuffle1_160000_filtered.csv",
    "post2026-05": "/media/toor/Seagate3/Mouse944/Round1/Linear/2026-06-26T15_27_25/dlc/Linear2026-06-26T15_27_25DLC_resnet50_linearMay29shuffle1_160000_filtered.csv",
}
DLC_ERA = "aging2024"  # <-- change to "aging2024" or "post2026-05" to check those instead
dlc_path = DLC_PATH_OPTIONS[DLC_ERA]

# 2. Load DLC CSV with 3-row MultiIndex header (scorer, bodypart, coord)
dlc_df = pd.read_csv(dlc_path, header=[0, 1, 2])
print("DLC shape:", dlc_df.shape)
print("Columns example:", dlc_df.columns[:5])

# 3. List available bodyparts, with mean likelihood (skips the non-bodypart
# "bodyparts" header column some DLC exports carry)
bodyparts = sorted({col[1] for col in dlc_df.columns if len(col) == 3 and col[1] != "bodyparts"})
mean_likelihood = {}
for bp in bodyparts:
    lik_col = [c for c in dlc_df.columns if len(c) == 3 and c[1] == bp and str(c[2]).lower() == "likelihood"][0]
    mean_likelihood[bp] = pd.to_numeric(dlc_df[lik_col], errors="coerce").mean()

print("\nAvailable bodyparts (mean likelihood):")
for bp in bodyparts:
    print(f"  - {bp}: {mean_likelihood[bp]:.3f}")

# 4. Automatically use the bodypart with the highest average likelihood
#    (override by setting bodypart explicitly, e.g. bodypart = "miniscope_base")
bodypart = max(mean_likelihood, key=mean_likelihood.get)
print(f"\nUsing bodypart: {bodypart!r} (mean likelihood {mean_likelihood[bodypart]:.3f})")

# 5. Helper to find the x/y columns for this bodypart
def col_of(dlc_df, bodypart, coord):
    coord = coord.lower()
    for col in dlc_df.columns:
        if len(col) == 3 and col[1] == bodypart and str(col[2]).lower() == coord:
            return col
    raise KeyError(f"Could not find coord={coord!r} for bodypart={bodypart!r}")

x_col = col_of(dlc_df, bodypart, "x")
y_col = col_of(dlc_df, bodypart, "y")
lik_col = col_of(dlc_df, bodypart, "likelihood")

x_raw = pd.to_numeric(dlc_df[x_col], errors="coerce").to_numpy()
y_raw = pd.to_numeric(dlc_df[y_col], errors="coerce").to_numpy()
lik_raw = pd.to_numeric(dlc_df[lik_col], errors="coerce").to_numpy()

XY_raw = np.column_stack([x_raw, y_raw])

print("\nXY_raw shape:", XY_raw.shape)
print("x range:", np.nanmin(x_raw), "→", np.nanmax(x_raw))
print("y range:", np.nanmin(y_raw), "→", np.nanmax(y_raw))
print("likelihood range:", np.nanmin(lik_raw), "→", np.nanmax(lik_raw))


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd

def plot_with_rois(XY, port_rois, subsample=5, title="ROI test", ylim=None, xlim=None):
    """
    XY        : (N,2) array of x,y raw or cleaned positions
    port_rois : dict{name: (xmin, xmax, ymin, ymax)}
    ylim, xlim: optional (lo, hi) in data coordinates, to zoom into a region
                (e.g. ylim=(400, 1200)) instead of showing the whole frame.
    """
    XY = np.asarray(XY, float)
    fig, ax = plt.subplots(figsize=(6,6))

    # subsample to avoid heavy plotting
    idx = np.arange(0, len(XY), subsample)

    ax.scatter(XY[idx,0], XY[idx,1], s=2, alpha=0.3, label="trajectory")

    # draw each roi
    for name, (xmin, xmax, ymin, ymax) in port_rois.items():
        rect = Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, fill=False, linewidth=2, color="red")
        ax.add_patch(rect)
        cx, cy = (xmin + xmax) / 2, (ymin + ymax) / 2
        ax.text(cx, cy, name, color="red", ha="center", va="center", fontsize=10)

    ax.set_aspect("equal", adjustable="box")
    if xlim is not None:
        ax.set_xlim(*xlim)
    ax.set_ylim(*(ylim if ylim is not None else (XY[:,1].min(), XY[:,1].max())))
    ax.invert_yaxis()     # matches DLC coordinate system
    ax.set_title(title)
    plt.show()


def print_roi_coverage(XY, port_rois):
    """
    Prints how many frames fall within each ROI
    """
    XY = np.asarray(XY, float)
    n = len(XY)
    print(f"Total frames: {n}")

    for name, (xmin, xmax, ymin, ymax) in port_rois.items():
        inside = (XY[:,0] >= xmin) & (XY[:,0] <= xmax) & (XY[:,1] >= ymin) & (XY[:,1] <= ymax)
        print(f"{name:10s}: {inside.sum():6d} frames ({inside.mean()*100:5.2f} %)")


In [ ]:
# Example manually guessed ROIs (you will adjust these)
test_rois = {
    "left":  (185, 305, 615, 715),   # (xmin, xmax, ymin, ymax)
    "right": (1690, 1810, 560, 660),
}

# Plot
plot_with_rois(XY_raw, test_rois, subsample=1, title="ROI Tuning Plot", ylim=(400, 1200))

# Frame counts inside each ROI
print_roi_coverage(XY_raw, test_rois)


## Usage

1. Port ROIs are looked up per session from `PORT_ROIS_BY_ERA` via `era_for_session(session_dir)`
   (3 eras: aging2024 / pre-2026-05 / post-2026-05 -- see the comment above `PORT_ROIS_BY_ERA`). For a new
   rig/camera position, add a new era entry rather than editing `PORT_ROIS_DEFAULT` directly --
   that constant is now only a backward-compatible alias for the pre-2026-05 entry, used by the
   single-session example cell below.
2. Optionally adjust global parameters like `FPS`, `N_BINS`, etc.
3. Set `session_dir` to a folder that contains:
      - `dlc_trim_info.csv`
      - a `dlc/` subfolder with one DLC CSV matching `dlc_pattern`.
4. Run:
      port_rois = PORT_ROIS_BY_ERA[era_for_session(session_dir)]
      process_lineartrack_session(session_dir, port_rois=port_rois)

You will get:
  - one frame-wise CSV with positions, ports, and bin indices
  - one trials table
  - one long-format CSV with per-trial, binned positions

These are analogous to the TMaze pipeline outputs but tailored to a linear track.

%%
Example (edit and run):
s
